# Fermion Sign vs Particle Number (n)
Computes Fermi and Boson energies using the Baxter method estimators and the exact indistinguishable Boson recursion.
Exports the data to a CSV for sign vs n plots.

In [1]:
import math
import numpy as np
import pandas as pd
import numba
from baxter_method import log_Q_all
from propagator import p_funcs_zeta_1, p_funcs_lambda, p_funcs_gamma, p_funcs_k1, factor_calc_T, factor_calc_H

# ============================================================
# Boson Partition Function Numba Kernel
# ============================================================

@numba.njit
def logsumexp2(x, y):
    if x == -np.inf:
        return y
    if y == -np.inf:
        return x
    m = x if x > y else y
    return m + math.log(math.exp(x - m) + math.exp(y - m))

@numba.njit
def log_Z_B_all(n, b):
    """Compute log Z_B(k) for all k=0..n for indistinguishable bosons in a 2D harmonic trap."""
    log_Z = np.full(n + 1, -np.inf, dtype=np.float64)
    log_Z[0] = 0.0
    
    logb = math.log(b)
    log_Z1_arr = np.full(n + 1, -np.inf, dtype=np.float64)
    for k in range(1, n + 1):
        bk = math.exp(k * logb)
        log_Z1_arr[k] = k * logb - 2.0 * math.log1p(-bk)
        
    for i in range(1, n + 1):
        s = -np.inf
        for k in range(1, i + 1):
            term = log_Z1_arr[k] + log_Z[i - k]
            s = logsumexp2(s, term)
        log_Z[i] = s - math.log(i)
        
    return log_Z


In [2]:
# ============================================================
# Estimator Functions
# ============================================================

def get_b_val(tau_val, N_val, w_val):
    if N_val == 0:
        return math.exp(-w_val * abs(tau_val))
    eps = w_val * (tau_val / N_val)
    z1 = p_funcs_zeta_1(eps)
    u = math.acosh(z1) if z1 >= 1.0 else 0.0
    return math.exp(-N_val * u)

def compute_energies(n_max, tau, N, w):
    """
    Computes the Fermi and Boson energies using the Thermodynamic and Hamiltonian estimators.
    """
    # Evaluation points for finite difference
    step = 1e-5
    tau1 = tau - step / 2.0
    tau2 = tau + step / 2.0
    
    # b values
    b_tau1 = get_b_val(tau1, N, w)
    b_tau2 = get_b_val(tau2, N, w)
    
    # Helper to calculate step derivative
    def fd_diff(arr1, arr2):
        return (arr1 - arr2) / step
        
    # --- Fermions (Baxter Method) ---
    lq1_F = log_Q_all(n_max, b_tau1)
    lq2_F = log_Q_all(n_max, b_tau2)
    energystar_T_all_F = fd_diff(lq1_F, lq2_F)
    
    # 1-particle energy is the same for fermions and bosons (at n=1)
    energy1star_T = energystar_T_all_F[1]
    
    # --- Bosons ---
    lz1_B = log_Z_B_all(n_max, b_tau1)
    lz2_B = log_Z_B_all(n_max, b_tau2)
    energystar_T_all_B = fd_diff(lz1_B, lz2_B)
    
    # True b values without Trotter (or same if PA is intended for 1-particle)
    b_true_tau1 = math.exp(-w * tau1)
    b_true_tau2 = math.exp(-w * tau2)
    
    # But wait, in the notebook, energy1_T uses exact b, while energystar uses PA.
    # Let's just use exact b for 1-particle to match `BaxterMeth-opt.ipynb`
    lq1_1 = log_Q_all(1, b_true_tau1)
    lq2_1 = log_Q_all(1, b_true_tau2)
    energy1_T = fd_diff(lq1_1, lq2_1)[1]
    
    # --- Factor Calculations for Hamiltonian ---
    eps = w * tau / N if N > 0 else 0.0
    z1 = p_funcs_zeta_1(eps)
    lam = p_funcs_lambda(eps)
    gam = p_funcs_gamma(eps)
    
    eps_s = w * tau if N > 0 else 0.0
    z1_s = p_funcs_zeta_1(eps_s)
    lam_s = p_funcs_lambda(eps_s)
    gam_s = p_funcs_gamma(eps_s)
    
    fT_reg, fT_star = factor_calc_T(lam, gam, w, lam_s, gam_s)
    fH_reg, fH_star = factor_calc_H(lam, gam, w, lam_s, gam_s)
    
    # --- Final Thermodynamic Estimators ---
    energy_T_F = energy1_T + (energystar_T_all_F - energy1star_T)
    energy_T_B = energy1_T + (energystar_T_all_B - energy1star_T)
    
    # --- Final Hamiltonian Estimators ---
    energy1_H = energy1_T * (fH_reg / fT_reg)
    energy_H_F = energy1_H + (energystar_T_all_F - energy1star_T) * (fH_star / fT_star)
    energy_H_B = energy1_H + (energystar_T_all_B - energy1star_T) * (fH_star / fT_star)
    
    # Handle the n=0 case
    energy_T_F[0] = 0.0
    energy_T_B[0] = 0.0
    energy_H_F[0] = 0.0
    energy_H_B[0] = 0.0
    
    return energy_T_F, energy_T_B, energy_H_F, energy_H_B


In [ ]:
# ============================================================
# Main Execution
# ============================================================

# Parameters
d = 2
tau = 5.0
N = 0 # Use 0 for continuous exact
w = 1.0
n_max = 16

energy_T_F, energy_T_B, energy_H_F, energy_H_B = compute_energies(n_max, tau, N, w)

# Calculate signs: sgn = exp(-tau * (E_f - E_B))
# Using Thermodynamic Estimator:
sgn_T = np.exp(-tau * (energy_T_F - energy_T_B))

# Using Hamiltonian Estimator:
sgn_H = np.exp(-tau * (energy_H_F - energy_H_B))

# Create DataFrame
n_arr = np.arange(n_max + 1)
df = pd.DataFrame({
    'n': n_arr,
    'E_f_T': energy_T_F,
    'E_B_T': energy_T_B,
    'sgn_T': sgn_T,
    'E_f_H': energy_H_F,
    'E_B_H': energy_H_B,
    'sgn_H': sgn_H
})

# Filter out n=0 if desired, but good to keep it.
# df = df[df['n'] > 0]

# Display head
display(df.head(10))

# Save to CSV
csv_filename = f'sgn_vs_n_data_tau{tau}_N{N}_w{w}.csv'
df.to_csv(csv_filename, index=False)
print(f"Data successfully saved to {csv_filename}")


,n,E_f_T,E_B_T,sgn_T,E_f_H,E_B_H,sgn_H
0,0,0.000000,0.000000,1.000000e+00,0.000000,0.000000,1.000000e+00
1,1,1.013567,1.013567,1.000000e+00,1.211530,1.211530,1.000000e+00
2,2,2.709833,1.861969,1.441740e-02,8.208625,4.711187,2.543368e-08
3,3,4.435127,2.709975,1.794232e-04,15.325465,8.209210,3.525727e-16
4,4,6.960221,3.557973,4.093663e-08,25.741477,11.707203,3.349364e-31
5,5,9.514154,4.405971,8.085800e-12,36.276448,15.205196,1.755324e-46
6,6,12.109115,5.253970,1.300891e-15,46.980664,18.703189,3.946783e-62
7,7,15.458526,6.101968,4.813846e-21,60.796984,22.201182,1.549817e-84
8,8,18.847681,6.949966,1.460299e-26,74.777247,25.699175,2.681156e-107
9,9,22.260688,7.797965,3.931834e-32,88.855902,29.197168,2.835992e-130


Data successfully saved to sgn_vs_n_data_tau5.0_N4_w1.0.csv
